# Práctica 2 - Introducción a PyTorch
---
### Contenido
- Demos con **PyTorch**: creación e inicialización de tensores, shapes, dtypes y operaciones.
- Inferencia de neurona y perceptrón con **torch.Tensor**.
- Ejercicios para implementar y experimentar.
---
### Cómo usar este cuaderno
- Ejecuta las celdas en orden.
- Completa las líneas marcadas con **`# TODO`**.
- Usa las celdas de **tests** (`assert`) para autocorregirte.

In [ ]:
import time
import torch
import matplotlib.pyplot as plt

torch.manual_seed(7)

print("Torch:", torch.__version__)
cpu = torch.device("cpu")
cuda = torch.device("cuda") if torch.cuda.is_available() else None
print("CUDA disponible:", torch.cuda.is_available())
if cuda is not None:
    print("GPU:", torch.cuda.get_device_name(0))

# Parte A - Fundamentos de PyTorch

### Mini píldora teórica: `torch.Tensor` como tensor
En PyTorch, un **tensor** (`torch.Tensor`) es un array n-dimensional con:
- `shape` (dimensiones)
- `dtype` (tipo numérico)
- `device` (CPU o GPU)

## 1) Demo - Creación básica e inspección de tensores

Observa:
- cómo se crean tensores escalares, vectores y matrices,
- cómo se consulta `shape`, `dtype` y `device`,
- indexado y slicing.

In [ ]:
a = torch.tensor(2.5)
v = torch.tensor([3.0, -1.0, 0.25])
M = torch.tensor([[1.0, 0.0, 2.0],
                  [-3.0, 4.0, 0.5]])

print("a:", a, "| shape:", tuple(a.shape), "| dtype:", a.dtype, "| device:", a.device)
print("\nv:", v, "| shape:", tuple(v.shape), "| dtype:", v.dtype, "| device:", v.device)
print("\nM:\n", M, "\nshape:", tuple(M.shape), "| dtype:", M.dtype, "| device:", M.device)

print("\nIndexado / slicing:")
print("v[1] =", v[1])
print("M[0, 2] =", M[0, 2])
print("M[:, 2] =", M[:, 2])

## 2) Demo - Inicializaciones de tensores

PyTorch ofrece muchas formas de inicializar tensores sin escribir bucles:
- `torch.zeros`, `torch.ones`
- `torch.rand` (uniforme en [0,1))
- `torch.randn` (normal estándar)
- `torch.arange` (secuencia)
- `torch.linspace` (n puntos entre dos valores)

In [ ]:
z = torch.zeros((2, 4))
o = torch.ones((2, 4))
u = torch.rand((2, 4))
g = torch.randn((2, 4))
r = torch.arange(0, 10, 2)
l = torch.linspace(-1.0, 1.0, 5)

print("zeros:\n", z)
print("ones:\n", o)
print("rand:\n", u)
print("randn:\n", g)
print("arange:\n", r)
print("linspace:\n", l)

## 3) Ejercicio - Indexado, slicing y máscaras booleanas

Completa las funciones marcadas con ***TODO*** para practicar patrones comunes en DL:
- filtrar por condición
- sustituir valores
- contar elementos que cumplen una condición

In [ ]:
def filter_greater_than(x, threshold):
    return x[x > threshold]

def replace_negatives_with_zero(x):
    return torch.where(x < 0, 0, x)

def count_in_range(x, low, high):
    return ((x >= low) & (x <= high)).sum().item()

In [ ]:
# Pruebas
x = torch.tensor([-2.0, -0.5, 0.0, 0.3, 1.2, 4.0])
print("filter >", filter_greater_than(x, 0.25))
print("replace neg:", replace_negatives_with_zero(x))
print("count in [0,1]:", count_in_range(x, 0.0, 1.0))

In [ ]:
# Tests de autocorrección
x = torch.tensor([-2.0, -0.5, 0.0, 0.3, 1.2, 4.0])

out = filter_greater_than(x, 0.25)
assert torch.allclose(out, torch.tensor([0.3, 1.2, 4.0]))

y = replace_negatives_with_zero(x)
assert torch.allclose(y, torch.tensor([0.0, 0.0, 0.0, 0.3, 1.2, 4.0]))
assert torch.allclose(x, torch.tensor([-2.0, -0.5, 0.0, 0.3, 1.2, 4.0]))

c01 = count_in_range(x, 0.0, 1.0)
assert c01 == 2  # 0.0 y 0.3
print("✅ Tests superados")

## 4) Demo - Pasar tensores a GPU

**Regla de oro:** en una operación, los tensores deben estar en el **mismo device**.
- `.to(device)` mueve tensores entre CPU/GPU.

In [ ]:
x = torch.randn(3, 3)
print("x device:", x.device)

if cuda is not None:
    x_gpu = x.to(cuda)
    print("x_gpu device:", x_gpu.device)
else:
    print("No hay GPU disponible: seguimos en CPU.")

## 5) Demo - CPU vs GPU

### Mini píldora teórica: `torch.cuda.synchronize()`

La ejecución de operaciones en la GPU es **asíncrona** por defecto.

Cuando se ejecuta una instrucción de PyTorch en la GPU (como `A @ B`, multiplicación matricial), Python "lanza" la tarea a la cola de la GPU y continúa inmediatamente con la siguiente línea de código, sin esperar a que la GPU termine el cálculo.

Si se mide el tiempo sin `torch.cuda.synchronize()`, el cronómetro se detendría casi al instante (solo se mediría el tiempo que tarda la CPU en enviar la orden), dando un resultado falso de que fue rapidísimo. El `synchronize()` obliga a la CPU a esperar a que la GPU complete todas las tareas pendientes antes de parar el reloj.

⚠️ **Nota:** la GPU tiene *overhead* de lanzamiento. Para operaciones pequeñas puede no ganar.

In [ ]:
def time_matmul(device, n=768, iters=30, dtype=torch.float32):
    A = torch.randn(n, n, device=device, dtype=dtype)
    B = torch.randn(n, n, device=device, dtype=dtype)

    # Calentamiento: Ejecutar operación 1 vez antes de medir.
    # Crucial en GPUs para inicializar cachés sin que afecte a la medición.
    _ = A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(iters):
        _ = A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) / iters

In [ ]:
t_cpu = time_matmul(cpu)
print(f"CPU matmul avg: {t_cpu*1000:.2f} ms")

if cuda is not None:
    t_gpu = time_matmul(cuda)
    print(f"GPU matmul avg: {t_gpu*1000:.2f} ms")
    print(f"Mejora aproximada: {t_cpu/t_gpu:.2f}x")
else:
    print("GPU no disponible: no se puede comparar.")

## 6) Demo - Broadcasting

### Mini píldora teórica: ¿qué es broadcasting?
- Es una funcionalidad de PyTorch que permite realizar operaciones aritméticas elemento a elemento entre **tensores de diferentes formas**.
- Como regla general, 2 tensores son compatibles para el broadcasting si, comenzando desde la última dimensión (derecha) y avanzando hacia la izquierda, se cumple alguna de estas condiciones:
    * Las dimensiones son iguales.
    * Una de las dimensiones es 1.
- Formas incompatibles producen error.
- Comprobar shapes temprano con `assert` ahorra tiempo.

In [ ]:
z = torch.tensor([1.0, 2.0, 3.0])
b = 0.25
print("z + b =", z + b)

mask = z > 2.0
print("mask (bool):", mask, mask.dtype)

print("mask.int():", mask.int(), mask.int().dtype)
print("mask.to(float):", mask.to(torch.float32), mask.to(torch.float32).dtype)

---
# Parte B - Neurona y perceptrón con PyTorch

## 7) Ejercicio - Funciones de transferencia vectorizadas

En PyTorch no hace falta "reinventar" funciones: muchas están ya implementadas.
Aun así, es importante saber **cómo se expresan** con tensores.

Completa las funciones marcadas con ***TODO*** (deben aceptar `torch.Tensor` y devolver `torch.Tensor`).

In [ ]:
def step_torch(z, threshold=0.0, dtype=torch.int64):
    return (z > threshold).to(dtype)

def saturated_linear_torch(z, min_val=0.0, max_val=1.0):
    return torch.clamp(z, min_val, max_val)

def sigmoid_torch(z):
    return 1 / (1 + torch.exp(-z))

In [ ]:
# Pruebas
z_test = torch.tensor([-2.5, -0.5, 0.0, 0.8, 2.2])
print("step:", step_torch(z_test))
print("sat:", saturated_linear_torch(z_test, 0.0, 1.0))
print("sig:", sigmoid_torch(z_test))

In [ ]:
# Tests de autocorrección
z_test = torch.tensor([-2.5, -0.5, 0.0, 0.8, 2.2])

expected_step = torch.tensor([0, 0, 0, 1, 1], dtype=torch.int64)
out_step = step_torch(z_test)
assert torch.equal(out_step.to(torch.int64), expected_step)

out_sat = saturated_linear_torch(z_test, 0.0, 1.0)
assert torch.allclose(out_sat, torch.tensor([0.0, 0.0, 0.0, 0.8, 1.0]))

out_sig = sigmoid_torch(z_test)
assert out_sig.shape == z_test.shape
assert torch.all((out_sig > 0.0) & (out_sig < 1.0))
assert abs(sigmoid_torch(torch.tensor(0.0)).item() - 0.5) < 1e-12

print("✅ Tests superados")

## 8) Demo - Suma ponderada y producto matricial con `@`

En PyTorch:
- producto escalar: `w @ x` (si ambos son 1D)
- batch: `X @ w` (si `X` es 2D y `w` es 1D)

In [ ]:
x = torch.tensor([1.5, -2.0, 0.25])
w = torch.tensor([0.4, 0.1, -0.8])
b = 0.3

z1 = w @ x + b
print("Single-sample z:", z1.item())

X = torch.tensor([[1.0, 0.0, 0.0],
                  [0.0, 1.0, 1.0],
                  [2.0, 1.0, -1.0]])
z_batch = X @ w + b
print("Batch z:", z_batch, "| shape:", tuple(z_batch.shape))

## 9) Ejercicio - Forward de neurona (batch)

Implementa la función `neuron_forward_torch(X, w, b, activation)` marcada con ***TODO***:
- incluye `assert` para shapes
- devuelve `(y_hat, z)` ambos con shape `(N,)`



In [ ]:
def neuron_forward_torch(X, w, b, activation):
    """
    X: torch.Tensor (N, D)
    w: torch.Tensor (D,)
    b: float o tensor 0D
    activation: función vectorizada (torch -> torch)
    """
    assert X.shape[1] == w.shape[0]
    z = X @ w + b
    y_hat = activation(z)
    return y_hat, z

In [ ]:
# Pruebas
X = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]])
w = torch.tensor([1.2, -0.7])
b = -0.1

y_hat, z = neuron_forward_torch(X, w, b, activation=sigmoid_torch)
print("z:", z)
print("y_hat(sigmoid):", y_hat)

In [ ]:
# Tests de autocorrección
X = torch.tensor([[0.0,0.0],[0.0,1.0],[1.0,0.0],[1.0,1.0]])
w = torch.tensor([1.2, -0.7])
b = -0.1

y_hat, z = neuron_forward_torch(X, w, b, activation=sigmoid_torch)
assert z.shape == (X.shape[0],)
assert y_hat.shape == (X.shape[0],)
assert torch.allclose(z, X @ w + b)
assert torch.all((y_hat > 0.0) & (y_hat < 1.0))

y_step, z2 = neuron_forward_torch(X, w, b, activation=lambda zz: step_torch(zz, threshold=0.0, dtype=torch.int64))
assert torch.allclose(z2, z)
assert y_step.dtype == torch.int64
assert set(y_step.tolist()).issubset({0, 1})

print("✅ Tests superados")

## 10) Demo - Perceptrón como caso particular

In [ ]:
def perceptron_predict_torch(X, w, b, threshold=0.0):
    y_hat, _ = neuron_forward_torch(
        X, w, b,
        activation=lambda zz: step_torch(zz, threshold=threshold, dtype=torch.int64)
    )
    return y_hat

In [ ]:
X_logic = torch.tensor([[0.0, 0.0],
                        [0.0, 1.0],
                        [1.0, 0.0],
                        [1.0, 1.0]])

y_or  = torch.tensor([0, 1, 1, 1], dtype=torch.int64)
y_and = torch.tensor([0, 0, 0, 1], dtype=torch.int64)

w_or  = torch.tensor([1.0, 1.0])
b_or  = -0.4
w_and = torch.tensor([1.0, 1.0])
b_and = -1.6

preds_or  = perceptron_predict_torch(X_logic, w_or,  b_or)
preds_and = perceptron_predict_torch(X_logic, w_and, b_and)
print("OR :", preds_or,  "target:", y_or)
print("AND:", preds_and, "target:", y_and)
assert torch.equal(preds_or, y_or)
assert torch.equal(preds_and, y_and)

---
# Parte C - Funcionalidades básicas adicionales de PyTorch

## 11) Ejercicio - Reshape, view, flatten y dimensión batch

En DL, casi todo acaba reorganizándose por shapes:
- pasar de imagen (H, W) a vector (H*W,)
- pasar de batch (N, ...) a (N, D)
- aplanar con `flatten`

Completa las funciones marcadas con ***TODO***.

⚠️ **Nota:** algunas operaciones necesitan que el tensor sea contiguo.

In [ ]:
def flatten_features(X):
    return X.view(X.shape[0], -1)

def reshape_like_image(x, H, W):
    return x.view(H, W)

def add_channel_dim(images):
    return images.unsqueeze(1)

In [ ]:
# Pruebas
X = torch.randn(5, 2, 3)  # (N=5, 2, 3)
print("flatten_features shape:", flatten_features(X).shape)

x = torch.arange(12.0)  # 12 = 3*4
print("reshape_like_image:\n", reshape_like_image(x, 3, 4))

imgs = torch.randn(8, 28, 28)
print("add_channel_dim shape:", add_channel_dim(imgs).shape)

In [ ]:
# Tests de autocorrección
X = torch.randn(5, 2, 3)
Y = flatten_features(X)
assert Y.shape == (5, 6)
assert torch.allclose(Y[0], X[0].reshape(-1))

x = torch.arange(12.0)
img = reshape_like_image(x, 3, 4)
assert img.shape == (3, 4)
assert torch.allclose(img.reshape(-1), x)

imgs = torch.randn(8, 28, 28)
imgs2 = add_channel_dim(imgs)
assert imgs2.shape == (8, 1, 28, 28)
assert torch.allclose(imgs2[:,0], imgs)
print("✅ Tests superados")

## 12) Ejercicio - Concatenación y apilado

En PyTorch hay dos operaciones típicas:
- `torch.cat`: concatena sobre una dimensión existente (aumenta el tamaño de esa dimensión)
- `torch.stack`: crea una nueva dimensión (apila tensores)

Completa las funciones marcadas con ***TODO*** y comprueba shapes.

In [ ]:
def concat_batches(A, B, dim=0):
    return torch.cat((A, B), dim=dim)

def stack_as_channels(*tensors):
    return torch.stack(tensors, dim=0)

def interleave_features(x1, x2):
    # Para intercalar: x1 y x2 son (N, D), devolver (N, 2*D) con [x1_0, x2_0, x1_1, x2_1, ...]
    # Usar torch.stack en dim=2 para (N, D, 2), luego transpose y reshape
    stacked = torch.stack((x1, x2), dim=2)  # (N, D, 2)
    transposed = stacked.transpose(1, 2)  # (N, 2, D)
    return transposed.reshape(x1.shape[0], -1)  # (N, 2*D)

In [ ]:
# Pruebas
A = torch.randn(3, 4)
B = torch.randn(2, 4)
print("concat_batches shape:", concat_batches(A, B).shape)

a = torch.zeros(2, 2)
b = torch.ones(2, 2)
c = 2*torch.ones(2, 2)
print("stack_as_channels shape:", stack_as_channels(a, b, c).shape)

x1 = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0]])
x2 = torch.tensor([[10.0, 20.0],
                   [30.0, 40.0]])
print("interleave:\n", interleave_features(x1, x2))

In [ ]:
# Tests de autocorrección
A = torch.randn(3, 4)
B = torch.randn(2, 4)
C = concat_batches(A, B)
assert C.shape == (5, 4)
assert torch.allclose(C[:3], A)
assert torch.allclose(C[3:], B)

a = torch.zeros(2, 2)
b = torch.ones(2, 2)
c = 2*torch.ones(2, 2)
S = stack_as_channels(a, b, c)
assert S.shape == (3, 2, 2)
assert torch.allclose(S[0], a)
assert torch.allclose(S[1], b)
assert torch.allclose(S[2], c)

x1 = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0]])
x2 = torch.tensor([[10.0, 20.0],
                   [30.0, 40.0]])
out = interleave_features(x1, x2)
expected = torch.tensor([[1.0, 10.0, 2.0, 20.0],
                         [3.0, 30.0, 4.0, 40.0]])
assert torch.allclose(out, expected)
print("✅ Tests superados")

## 13) Ejercicio - Reducciones

En DL se usan reducciones constantemente:
- pérdidas: medias sobre batch
- normalizaciones y estadísticas
- obtener clases con `argmax`

Completa las funciones marcadas con ***TODO*** prestando atención a `dim` y `keepdim`.

In [ ]:
def batch_mean(X):
    return X.mean(dim=0)

def rowwise_l2_norm(X, eps=1e-12):
    return torch.sqrt((X**2).sum(dim=1) + eps)

def argmax_classes(scores):
    return scores.argmax(dim=1)

In [ ]:
# Pruebas
X = torch.tensor([[1.0, 2.0, 3.0],
                  [2.0, 0.0, 1.0],
                  [0.0, 2.0, 2.0]])
print("batch_mean:", batch_mean(X))
print("rowwise_l2_norm:", rowwise_l2_norm(X))

scores = torch.tensor([[0.1, 0.9, 0.0],
                       [2.0, 1.0, 3.0],
                       [-1.0, -2.0, -0.5]])
print("classes:", argmax_classes(scores))

In [ ]:
# Tests de autocorrección
X = torch.tensor([[1.0, 2.0, 3.0],
                  [2.0, 0.0, 1.0],
                  [0.0, 2.0, 2.0]])
m = batch_mean(X)
assert m.shape == (3,)
assert torch.allclose(m, torch.tensor([1.0, 4.0/3.0, 2.0]))

n = rowwise_l2_norm(X)
assert n.shape == (3,)
expected = torch.sqrt(torch.tensor([1.0**2+2.0**2+3.0**2, 2.0**2+0.0**2+1.0**2, 0.0**2+2.0**2+2.0**2]))
assert torch.allclose(n, expected)

scores = torch.tensor([[0.1, 0.9, 0.0],
                       [2.0, 1.0, 3.0],
                       [-1.0, -2.0, -0.5]])
cls = argmax_classes(scores)
assert torch.equal(cls, torch.tensor([1, 2, 2]))
print("✅ Tests superados")